In [1]:
import json
import pandas as pd
import os 
with open("raw_studies.json") as f:
    all_studies = json.load(f)
print(len(all_studies))

3721


In [15]:
# Sites table

site_rows = []
for s in all_studies:
    ps=s["protocolSection"]
    nct=ps["identificationModule"]["nctId"]
    for loc in ps.get("contactsLocationsModule",{}).get("locations",{}):
        row={"nctId":nct}
        row.update(loc)
        site_rows.append(row)
df_sites_all=pd.json_normalize(site_rows)
df_sites_all=df_sites_all.drop(columns="contacts",errors="ignore")
df_sites_ro=df_sites_all[df_sites_all["country"]=="Romania"]
ro_nct_ids=df_sites_ro["nctId"].unique()

print(df_sites_all.shape)
print(df_sites_all["nctId"].nunique())
print(df_sites_ro.shape)
print(df_sites_ro["nctId"].nunique())


(449289, 9)
3721
(16949, 9)
3720


In [16]:
# Studies table

study_rows = []
for s in all_studies:
    ps = s["protocolSection"]
    study = {
        "nctId": ps["identificationModule"]["nctId"],
        "studyType": ps.get("designModule",{}).get("studyType"),
        "phases": ps.get("designModule",{}).get("phases"),
        "status": ps.get("statusModule",{}).get("overallStatus"),
        "startDate" : ps.get("statusModule",{}).get("startDateStruct",{}).get("date"),
        "startType" : ps.get("statusModule",{}).get("startDateStruct",{}).get("type"), 
        "sponsor" : ps.get("sponsorCollaboratorsModule", {}).get("leadSponsor",{}).get("name"),
        "sponsorClass" : ps.get("sponsorCollaboratorsModule", {}).get("leadSponsor",{}).get("class")
    }
    study_rows.append(study)

df_study = pd.DataFrame(study_rows)
print(df_study.shape)
print(df_study["nctId"].nunique())
print(df_study["studyType"].value_counts(dropna=False))
print(df_study.isna().sum())

(3721, 8)
3721
studyType
INTERVENTIONAL     3294
OBSERVATIONAL       416
EXPANDED_ACCESS      11
Name: count, dtype: int64
nctId              0
studyType          0
phases           428
status             0
startDate         10
startType       1281
sponsor            0
sponsorClass       0
dtype: int64


In [4]:
# Filtres and funnel

df_ro_study = df_study[df_study["nctId"].isin(ro_nct_ids)].copy()
df_ro_study["startYear"] = pd.to_numeric(df_ro_study["startDate"].str[:4])
df_2015 = df_ro_study[df_ro_study["startYear"] >= 2015].copy() # N
df_int = df_2015[df_2015["studyType"]== "INTERVENTIONAL"].copy() # M
print(len(df_study), len(df_ro_study), len(df_2015), len(df_int))

3721 3720 1984 1717


In [5]:
# Phases

phase_map = {
    "EARLY_PHASE1":  "Early Phase 1",
    "PHASE1":        "Phase 1",
    "PHASE1/PHASE2": "Phase 1/2",
    "PHASE2":        "Phase 2",
    "PHASE2/PHASE3": "Phase 2/3",
    "PHASE3":        "Phase 3",
    "PHASE4":        "Phase 4",
    "NA":            "No phase",
}
def phase_label(phases):
    if not isinstance(phases,list) or len(phases) == 0:
        return "No phase"
    return phase_map.get("/".join(phases), "Other")
df_int["phaseLabel"] = df_int["phases"].apply(phase_label)

print(df_int["phaseLabel"].value_counts(dropna=False))

phaseLabel
Phase 3          929
Phase 2          300
No phase         287
Phase 1           68
Phase 2/3         51
Phase 4           48
Phase 1/2         33
Early Phase 1      1
Name: count, dtype: int64


In [6]:
print(df_int["phases"].dropna().apply(lambda p: "/".join(p)).value_counts())

phases
PHASE3           929
PHASE2           300
NA               286
PHASE1            68
PHASE2/PHASE3     51
PHASE4            48
PHASE1/PHASE2     33
EARLY_PHASE1       1
Name: count, dtype: int64


In [7]:
# 3rd decision - type of startDate, after 2015

print(df_2015["startType"].value_counts(dropna=False))

startType
ACTUAL       1850
ESTIMATED      70
None           64
Name: count, dtype: int64


In [8]:
# those with no startDate in Ro

fara_tip = df_ro_study[df_ro_study["startType"].isna()]
print (len(fara_tip))
print(fara_tip["startYear"].describe())

1280
count    1270.000000
mean     2009.066142
std         3.260969
min      1994.000000
25%      2007.000000
50%      2009.000000
75%      2011.000000
max      2017.000000
Name: startYear, dtype: float64


In [9]:
# 4th decision - final types of status

print(df_int["status"].value_counts(dropna=False))

status
COMPLETED                  795
RECRUITING                 346
ACTIVE_NOT_RECRUITING      291
TERMINATED                 167
UNKNOWN                     67
NOT_YET_RECRUITING          24
ENROLLING_BY_INVITATION     15
WITHDRAWN                   10
SUSPENDED                    2
Name: count, dtype: int64


In [10]:
# startTypes

print(df_2015.groupby("startType", dropna=False)["startYear"].agg(["count", "min", "max"]))

           count     min     max
startType                       
ACTUAL      1850  2015.0  2026.0
ESTIMATED     70  2017.0  2027.0
NaN           64  2015.0  2017.0


In [11]:
# no startType studies

fara_tip_2015 = df_2015[df_2015["startType"].isna()]
print(fara_tip_2015["status"].value_counts())

est = df_2015[df_2015["startType"] == "ESTIMATED"]
print(est["startYear"].value_counts().sort_index())

status
COMPLETED     47
UNKNOWN        9
TERMINATED     7
WITHDRAWN      1
Name: count, dtype: int64
startYear
2017.0     2
2018.0     1
2019.0     2
2020.0     5
2021.0     7
2022.0     1
2023.0     3
2024.0     7
2025.0    11
2026.0    30
2027.0     1
Name: count, dtype: int64


In [12]:
# startKind

def start_kind(row):
    if row["startType"] == "ACTUAL":
        return "actual"
    if row["startType"] == "ESTIMATED":
        return "planned"
    if row["status"] in ["COMPLETED", "TERMINATED"]:
        return "actual"
    return "unknown"

df_int["startKind"] = df_int.apply(start_kind, axis=1)
print(df_int["startKind"].value_counts())

startKind
actual     1663
planned      45
unknown       9
Name: count, dtype: int64


In [13]:
# not updated records

print((df_int.loc[df_int["startKind"] == "planned", "startYear"] < 2026).sum()) # X

23


In [14]:
# saving table

cols = ["nctId", "studyType", "phaseLabel","status","startDate","startType", "startYear","startKind", "sponsor", "sponsorClass"]
out_dir = "../data/clean"
os.makedirs(out_dir,exist_ok=True)

df_int[cols].to_csv(f"{out_dir}/studies.csv", index=False)

check = pd.read_csv(f"{out_dir}/studies.csv")
print(check.shape) 

(1717, 10)
